In [1]:
from langgraph.store.memory import InMemoryStore

In [4]:
# creating a store
store = InMemoryStore()

In [3]:
# creating a namespace
namespace = ("user", "u1")

### Creating Memories

In [5]:
store.put(namespace, "key1", {"data": "User likes pizza"})
store.put(namespace, "key2", {"data": "User prefers dark mode"})

In [6]:
# another namespace
namespace2 = ("user", "u2")

# Add memories
store.put(namespace2, "1", {"data": "User likes pasta"})
store.put(namespace2, "2", {"data": "User prefers grid style navigation"})

### Retrieving Memories

In [7]:
store.get(namespace, "key1")

Item(namespace=['user', 'u1'], key='key1', value={'data': 'User likes pizza'}, created_at='2026-08-20T09:12:29.332283+00:00', updated_at='2026-08-20T09:12:29.332285+00:00')

In [9]:
store.get(namespace2, "2")

Item(namespace=['user', 'u2'], key='2', value={'data': 'User prefers grid style navigation'}, created_at='2026-08-20T09:12:36.414856+00:00', updated_at='2026-08-20T09:12:36.414857+00:00')

### Retrieving all Memories

In [11]:
items = store.search(namespace)

for item in items:
    print(item)

Item(namespace=['user', 'u1'], key='key1', value={'data': 'User likes pizza'}, created_at='2026-08-20T09:12:29.332283+00:00', updated_at='2026-08-20T09:12:29.332285+00:00', score=None)
Item(namespace=['user', 'u1'], key='key2', value={'data': 'User prefers dark mode'}, created_at='2026-08-20T09:12:29.332327+00:00', updated_at='2026-08-20T09:12:29.332328+00:00', score=None)


### Semantic Search

In [12]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

Here we re-create the store, but this time we pass an `index` config so it can do **semantic search** instead of only exact-key lookups.

- `embed` — the embedding model used to turn memory text into a vector. It runs on `put()` (to index each memory) and on `search()` (to embed the query), so memories get matched by *meaning* rather than by key.
- `dims` — the length of those vectors. It has to match the model: `text-embedding-3-small` outputs 1536 dimensions.

Why we need it: with the plain `InMemoryStore()` above, `store.get()` needs the exact key and `store.search(namespace)` just dumps everything. That falls apart for real long-term memory, where the agent asks things like *"what is the user currently learning"* without knowing which key holds the answer. With indexing on, `store.search(namespace, query="...")` embeds the question and returns the closest memories ranked by similarity.

Note this rebinds `store` to a fresh, empty instance — the memories added earlier are gone, so we add a new set below.

In [13]:
store = InMemoryStore(index={'embed': embedding_model, 'dims':1536})

In [14]:
namespace = ('users', 'u1')

In [18]:
store.put(namespace, "1", {"data": "User prefers concise answers over long explanations"})
store.put(namespace, "2", {"data": "User likes examples in Python"})
store.put(namespace, "3", {"data": "User usually works late at night"})
store.put(namespace, "4", {"data": "User prefers dark mode in applications"})
store.put(namespace, "5", {"data": "User is learning machine learning"})
store.put(namespace, "6", {"data": "User dislikes overly theoretical explanations"})
store.put(namespace, "7", {"data": "User prefers step-by-step reasoning"})
store.put(namespace, "8", {"data": "User is based in Pakistan"})
store.put(namespace, "9", {"data": "User likes real-world analogies"})
store.put(namespace, "10", {"data": "User prefers bullet points over paragraphs"})

In [19]:
items = store.search(namespace, query="what is the user currently learning", limit=1)

for item in items:
    print(item.value)

{'data': 'User is learning machine learning'}


In [20]:
items = store.search(namespace, query="what are user's preferences", limit=3)

for item in items:
    print(item.value)

{'data': 'User prefers dark mode in applications'}
{'data': 'User prefers bullet points over paragraphs'}
{'data': 'User prefers concise answers over long explanations'}
